# Importar os dados de Cork Stoppers

In [9]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import os

# Ler o ficheiro CSV: separador ';', decimal ',' e saltar a linha vazia (linha 2)
df = pd.read_csv(
    r"C:/Users/josem/OneDrive/Ambiente de Trabalho/Zé Miguel/UNIVERSIDADE/5ANO/Rodrigo_Jose/AC/datasets/CORK_STOPPERS.csv",
    sep=";",
    decimal=",",
    skiprows=[1]
)

# Selecionar as colunas pretendidas e remover linhas com valores em falta
CorkStopp = df[["C", "ART", "N.1", "PRT", "ARM", "PRM",
                "ARTG", "NG", "PRTG", "RAAR", "RAN"]].dropna().copy()

# Nomes das features, excluindo a primeira coluna, que contém a classe
fnames = CorkStopp.columns[1:]

# Obter os índices das amostras de cada classe
ixSuper = np.where(CorkStopp["C"] == 1)[0]
ixAverage = np.where(CorkStopp["C"] == 2)[0]

# Verificar os resultados
print("Dimensões do dataset:", CorkStopp.shape)
print("Features:", list(fnames))
print("Número de amostras da classe 1:", len(ixSuper))
print("Número de amostras da classe 2:", len(ixAverage))

Dimensões do dataset: (150, 11)
Features: ['ART', 'N.1', 'PRT', 'ARM', 'PRM', 'ARTG', 'NG', 'PRTG', 'RAAR', 'RAN']
Número de amostras da classe 1: 50
Número de amostras da classe 2: 50


# Preparar o espaço para aplicar as métricas

In [10]:
# Extract the 'N.1' and 'PRT' features for the "Super" and "Average" classes
# using the indices obtained from the previous step

# Extract the 'N.1' feature for the "Super" class
NSup = CorkStopp['N.1'].to_numpy()[ixSuper]

# Extract the 'N.1' feature for the "Average" class
NAvg = CorkStopp['N.1'].to_numpy()[ixAverage]

# Extract the 'PRT' feature, divide it by 10, and select the "Super" class
PRT10Sup = (CorkStopp['PRT'].to_numpy() / 10)[ixSuper]

# Extract the 'PRT' feature, divide it by 10, and select the "Average" class
PRT10Avg = (CorkStopp['PRT'].to_numpy() / 10)[ixAverage]

# Calculate the mean vector for the "Super" class
# The vector contains the mean of 'N.1' and the mean of 'PRT/10'
# Transpose the array to obtain a column vector (2x1)
muSup = np.array([[np.mean(NSup), np.mean(PRT10Sup)]]).T

# Calculate the mean vector for the "Average" class
# The vector contains the mean of 'N.1' and the mean of 'PRT/10'
# Transpose the array to obtain a column vector (2x1)
muAvg = np.array([[np.mean(NAvg), np.mean(PRT10Avg)]]).T


# Print the linear discriminant function for the "Super" class
# gSup(x) = muSup.T @ x - 0.5 * (muSup.T @ muSup)
print("gSup(x)=" + str(muSup.T) + "x-0.5" + str(muSup.T @ muSup))

# Print the linear discriminant function for the "Average" class
# gAvg(x) = muAvg.T @ x - 0.5 * (muAvg.T @ muAvg)
print("gAvg(x)=" + str(muAvg.T) + "x-0.5" + str(muAvg.T @ muAvg))

gSup(x)=[[55.28  36.504]]x-0.5[[4388.420416]]
gAvg(x)=[[79.74  64.284]]x-0.5[[10490.900256]]


g1(x)=g2(x)

In [11]:
print("d(x)="+str((muSup-muAvg).T)+"x-0.5"+str((muSup-muAvg).T@(muSup+muAvg)))

d(x)=[[-24.46 -27.78]]x-0.5[[-6102.47984]]


# Extrair as métricas

In [13]:
# Create a feature matrix for the "Super" class
# Each row represents one sample and each column represents one feature
X1 = np.array([NSup, PRT10Sup]).T

# Create a feature matrix for the "Average" class
# Each row represents one sample and each column represents one feature
X2 = np.array([NAvg, PRT10Avg]).T

# Combine the samples from both classes into a single dataset
# axis=0 concatenates the matrices along the rows
X = np.concatenate((X1, X2), axis=0)

# Create the vector of true class labels by combining the labels
# of the "Super" and "Average" samples in the same order as X
y = np.concatenate((
    CorkStopp['C'].to_numpy()[ixSuper],
    CorkStopp['C'].to_numpy()[ixAverage]
))

# Initialize the predicted labels vector
# All samples are initially assigned to class 1 ("Super")
yp = np.ones(np.shape(y))


# Calculate the discriminant function for every sample
# d(x) = gSup(x) - gAvg(x)
# A positive value indicates "Super" (class 1)
# A negative value indicates "Average" (class 2)
dx = (
    (muSup - muAvg).T
    @ (X.T - 0.5 * (muSup + muAvg))
).flatten()

# Assign samples with a negative discriminant value to class 2 ("Average")
yp[dx < 0] = 2


# Calculate the number of correctly classified samples
# A hit occurs when the predicted label matches the true label
Hits = np.shape(np.where(y == yp)[0])[0]

# Calculate True Positives (TP)
# Samples belonging to "Super" that were correctly classified as "Super"
TP = np.shape(np.where(y[ixSuper] == yp[ixSuper])[0])[0]

# Calculate True Negatives (TN)
# Samples belonging to "Average" that were correctly classified as "Average"
TN = np.shape(np.where(y[ixAverage] == yp[ixAverage])[0])[0]

# Calculate False Positives (FP)
# Samples belonging to "Average" that were incorrectly classified as "Super"
FP = np.shape(np.where(y[ixAverage] != yp[ixAverage])[0])[0]

# Calculate False Negatives (FN)
# Samples belonging to "Super" that were incorrectly classified as "Average"
FN = np.shape(np.where(y[ixSuper] != yp[ixSuper])[0])[0]


# Calculate Sensitivity (Recall)
# Measures the proportion of "Super" samples correctly identified
SS = TP / (TP + FN)

# Calculate Specificity
# Measures the proportion of "Average" samples correctly identified
SP = TN / (TN + FP)

# Calculate Precision
# Measures the proportion of predicted "Super" samples that are actually "Super"
PR = TP / (TP + FP)

# Calculate the F1-score
# Harmonic mean of Precision and Sensitivity
F1Score = 2 * (PR * SS) / (PR + SS)

# Calculate Accuracy
# Measures the proportion of all samples correctly classified
AC = (TN + TP) / (TP + TN + FP + FN)


# Print the performance metrics as percentages
print("Sensitivity(%)=" + str(SS * 100))
print("Specificity(%)=" + str(SP * 100))
print("Precision(%)=" + str(PR * 100))
print("F1Score(%)=" + str(F1Score * 100))
print("Accuracy(%)=" + str(AC * 100))

Sensitivity(%)=84.0
Specificity(%)=78.0
Precision(%)=79.24528301886792
F1Score(%)=81.55339805825243
Accuracy(%)=81.0


-Represent geometrically the samples, class means, and decision hyperplane?

In [ ]:
# Create a new Plotly figure
fig = go.Figure()

# Plot the samples belonging to the "Super" class
# N is plotted on the x-axis and PRT/10 on the y-axis
fig.add_trace(go.Scatter(
    x=NSup, y=PRT10Sup, name='Super'
))

# Optional: customize the marker color and size
# fig.update_traces(marker=dict(color='green', size=10))

# Plot the samples belonging to the "Average" class
fig.add_trace(go.Scatter(
    x=NAvg, y=PRT10Avg, name='Average'
))

# Set the marker size for all traces added so far
fig.update_traces(marker=dict(size=10))


# Identify the indices of incorrectly classified "Average" samples
# Add the number of "Super" samples to obtain their positions in X
ixFP = np.where(y[ixAverage] != yp[ixAverage])[0] + 50

# Identify the indices of incorrectly classified "Super" samples
ixFN = np.where(y[ixSuper] != yp[ixSuper])[0]

# Highlight the incorrectly classified "Super" samples
# These are false negatives: "Super" samples predicted as "Average"
fig.add_trace(go.Scatter(
    x=X[ixFN, 0],
    y=X[ixFN, 1],
    name='False Average',
    marker_size=15,
    marker_symbol="circle-open",
    marker=dict(
        color="red",
        line=dict(width=3, color='red')
    )
))

# Highlight the incorrectly classified "Average" samples
# These are false positives: "Average" samples predicted as "Super"
fig.add_trace(go.Scatter(
    x=X[ixFP, 0],
    y=X[ixFP, 1],
    name='False Super',
    marker_size=15,
    marker_symbol="circle-open",
    marker=dict(
        color="blue",
        line=dict(width=3, color='blue')
    )
))


# Plot the mean vector of the "Super" class
# The mean is represented by a green cross
fig.add_trace(go.Scatter(
    x=muSup[0],
    y=muSup[1],
    marker_size=20,
    marker_symbol='x',
    name='Super mean',
    marker=dict(
        color="green",
        line=dict(width=1, color='black')
    )
))

# Plot the mean vector of the "Average" class
# The mean is represented by a yellow cross
fig.add_trace(go.Scatter(
    x=muAvg[0],
    y=muAvg[1],
    marker_size=20,
    marker_symbol='x',
    name='Average mean',
    marker=dict(
        color="yellow",
        line=dict(width=1, color='black')
    )
))

# Calculate and plot the midpoint between the two class means
# This point is the arithmetic mean of the two mean vectors
fig.add_trace(go.Scatter(
    x=(muSup[0] + muAvg[0]) / 2,
    y=(muSup[1] + muAvg[1]) / 2,
    marker_size=20,
    marker_symbol="circle-dot",
    name='Mean of the means',
    marker=dict(
        color="purple",
        line=dict(width=1, color='black')
    )
))


# Set all traces added so far to display markers
fig.update_traces(mode='markers')

# Draw a line segment connecting the two class mean vectors
fig.add_trace(go.Scatter(
    x=[muSup[0][0], muAvg[0][0]],
    y=[muSup[1][0], muAvg[1][0]],
    mode='lines',
    line=dict(color="black", width=4),
    name='Inter mean segment'
))

# Calculate the normal vector of the decision boundary
# W is the difference between the two class mean vectors
W = muSup - muAvg

# Calculate the constant term of the linear discriminant function
# d(x) = W.T @ x + b
b = -0.5 * (muSup - muAvg).T @ (muSup + muAvg)

# Generate x-axis values for plotting the decision boundary
# The range from 20 to 129 defines the horizontal extent of the line
x1 = np.arange(20, 130, 1)

# Calculate the corresponding y-axis values of the decision boundary
# The equation is obtained by setting d(x) = 0 and solving for x2
# Here, x1 represents N and x2 represents PRT/10
x2 = -(
    (muSup[0][0] - muAvg[0][0])
    / (muSup[1][0] - muAvg[1][0])
) * x1 + 0.5 * (
    muSup[0][0]**2 + muSup[1][0]**2
    - muAvg[0][0]**2 - muAvg[1][0]**2
) / (muSup[1][0] - muAvg[1][0])

# Plot the decision boundary as a dashed gray line
# This line separates the regions assigned to the two classes
fig.add_trace(go.Scatter(
    x=x1,
    y=x2.flatten(),
    mode='lines',
    line=dict(dash='dash', color="gray", width=4),
    name='Hyperplane'
))


# Label the x-axis and y-axis
fig.update_xaxes(title_text='N')
fig.update_yaxes(title_text='PRT/10')

# Configure the figure dimensions
fig.update_layout(
    autosize=False,
    width=900,
    height=800
)

# Display the final figure
fig.show()

c:\Users\josem\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




# O mesmo para Mahalanobis

In [15]:
# Extract the 'N.1' feature for the "Super" class
NSup = CorkStopp['N.1'].to_numpy()[ixSuper]
# Extract the 'N.1' feature for the "Average" class
NAvg = CorkStopp['N.1'].to_numpy()[ixAverage]
# Extract the 'PRT' feature, divide it by 10, and select the "Super" class
PRT10Sup = (CorkStopp['PRT'].to_numpy() / 10)[ixSuper]
# Extract the 'PRT' feature, divide it by 10, and select the "Average" class
PRT10Avg = (CorkStopp['PRT'].to_numpy() / 10)[ixAverage]
# Calculate the mean vector for the "Super" class
# The vector contains the mean values of 'N.1' and 'PRT/10'
# Transpose the array to obtain a column vector (2x1)
muSup = np.array([[np.mean(NSup), np.mean(PRT10Sup)]]).T
# Calculate the mean vector for the "Average" class
# The vector contains the mean values of 'N.1' and 'PRT/10'
# Transpose the array to obtain a column vector (2x1)
muAvg = np.array([[np.mean(NAvg), np.mean(PRT10Avg)]]).T


# Compute the covariance matrix for the "Super" class
# The rows represent the two features: 'N.1' and 'PRT/10'
# The covariance matrix describes their variances and covariance
CSup = np.cov(np.array([NSup, PRT10Sup]))

# Compute the covariance matrix for the "Average" class
CAvg = np.cov(np.array([NAvg, PRT10Avg]))

# Estimate the common covariance matrix by averaging the two class covariance matrices
C = (CSup + CAvg) / 2

# Calculate the inverse of the common covariance matrix
# This inverse is used in the linear discriminant functions
Ci = np.linalg.inv(C)

# Print the inverse covariance matrix
print(Ci)

# Print the linear discriminant function for the "Super" class
# gSup(x) = muSup.T @ Ci @ x - 0.5 * muSup.T @ Ci @ muSup
print("gSup(x)=" + str(muSup.T @ Ci) + "x-0.5" + str(muSup.T @ Ci @ muSup))

# Print the linear discriminant function for the "Average" class
# gAvg(x) = muAvg.T @ Ci @ x - 0.5 * muAvg.T @ Ci @ muAvg
print("gAvg(x)=" + str(muAvg.T @ Ci) + "x-0.5" + str(muAvg.T @ Ci @ muAvg))

[[ 0.0216046  -0.02555068]
 [-0.02555068  0.03601284]]
gSup(x)=[[ 0.26160002 -0.09782884]]x-0.5[[10.89010522]]
gAvg(x)=[[0.08025053 0.27763826]]x-0.5[[24.24687549]]


g1(x)=g2(x)

In [16]:
print("d(x)="+str((muSup-muAvg).T@Ci)+"x-0.5"+str((muSup-muAvg).T@Ci@(muSup+muAvg))) #

d(x)=[[ 0.18134949 -0.37546711]]x-0.5[[-13.35677027]]


# Compute the metrics

In [17]:
X1=np.array([NSup,PRT10Sup]).T
X2=np.array([NAvg,PRT10Avg]).T
X=np.concatenate((X1,X2),axis=0)
y=np.concatenate((CorkStopp['C'].to_numpy()[ixSuper],CorkStopp['C'].to_numpy()[ixAverage]))

yp=np.ones(np.shape(y))

dx=((muSup-muAvg).T@Ci@(X.T-0.5*(muSup+muAvg))).flatten() # d(i,j) = gi(x) - gj(x) - > if d(i,j) > 0 then assign to class i, else assign to class j
yp[dx<0]=2

Hits=np.shape(np.where((y==yp))[0])[0]
TP=np.shape(np.where((y[ixSuper]==yp[ixSuper]))[0])[0]
TN=np.shape(np.where((y[ixAverage]==yp[ixAverage]))[0])[0]
FP=np.shape(np.where((y[ixAverage]!=yp[ixAverage]))[0])[0]
FN=np.shape(np.where((y[ixSuper]!=yp[ixSuper]))[0])[0]

SS=TP/(TP+FN)
SP=TN/(TN+FP)
PR=TP/(TP+FP)
F1Score=2*(PR*SS)/(PR+SS)
AC=(TN+TP)/(TP+TN+FP+FN)


print("Sensitivity(%)="+str(SS*100))
print("Specificity(%)="+str(SP*100))
print("Precision(%)="+str(PR*100))
print("F1Score(%)="+str(F1Score*100))
print("Accuracy(%)="+str(AC*100))



Sensitivity(%)=98.0
Specificity(%)=82.0
Precision(%)=84.48275862068965
F1Score(%)=90.74074074074075
Accuracy(%)=90.0


plot final 

In [18]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=NSup,y=PRT10Sup,name='Super'))
fig.add_trace(go.Scatter(x=NAvg,y=PRT10Avg,name='Average'))
fig.update_traces(marker=dict(size=10))


ixFP=np.where((y[ixAverage]!=yp[ixAverage]))[0]+50
ixFN=np.where((y[ixSuper]!=yp[ixSuper]))[0]

fig.add_trace(go.Scatter(x=X[ixFN,0],y=X[ixFN,1],name='False Average',marker_size=15,
                         marker_symbol="circle-open",marker=dict(color="red",
                                                                 line=dict(width=3,color='red'))))
fig.add_trace(go.Scatter(x=X[ixFP,0],y=X[ixFP,1],name='False Super',marker_size=15,
                         marker_symbol="circle-open",marker=dict(color="blue",
                                                                 line=dict(width=3,color='blue'))))
fig.add_trace(go.Scatter(x=muSup[0],y=muSup[1], marker_size=20,
                         marker_symbol='x',name='Super mean',
                         marker=dict(color="green",line=dict(width=1,
                                        color='black'))))
fig.add_trace(go.Scatter(x=muAvg[0],y=muAvg[1], marker_size=20,
                         marker_symbol='x',name='Average mean',
                         marker=dict(color="yellow",line=dict(width=1,
                                        color='black'))))
fig.add_trace(go.Scatter(x=(muSup[0]+muAvg[0])/2,y=(muSup[1]+muAvg[1])/2, marker_size=20,
                         marker_symbol="circle-dot",name='Mean of the means',
                         marker=dict(color="purple",line=dict(width=1,
                                        color='black'))))

fig.update_traces(mode='markers')
fig.add_trace(go.Scatter(x=[muSup[0][0],muAvg[0][0]],y=[muSup[1][0],muAvg[1][0]],
                         mode='lines', line=dict(color="black",width=4),name='Inter mean segment'))


W=((muSup-muAvg).T@Ci).T
b=-0.5*(muSup-muAvg).T@Ci@(muSup+muAvg)

x1=np.arange(20,130,1)

x2=-(W[0,0]/W[1,0])*x1-b/W[1,0]
fig.add_trace(go.Scatter(x=x1,y=x2.flatten(),
                         mode='lines', line=dict(dash='dash',color="gray",width=4),name='Hyperplane'))

fig.update_xaxes(title_text='N')
fig.update_yaxes(title_text='PRT/10')
fig.update_layout(
    autosize=False,
    width=900,
    height=800)
fig.show()

# FISHER Linear Discriminant

In [19]:
from sklearn.model_selection import train_test_split
# Create the feature matrix for the "Super" class
# Each row represents one cork, and each column represents one feature
X1 = np.array([NSup, PRT10Sup]).T

# Create the feature matrix for the "Average" class
X2 = np.array([NAvg, PRT10Avg]).T

# Combine the samples from both classes into a single dataset
X = np.concatenate((X1, X2), axis=0)

# Create the target vector containing the class labels
# Each label corresponds to the respective sample in X
y = np.concatenate((
    CorkStopp['C'].to_numpy()[ixSuper],
    CorkStopp['C'].to_numpy()[ixAverage]
))

# Determine the total number of samples in the dataset
nSamp = np.shape(X)[0]

# Calculate the number of samples to use for training (70%)
nSampTr = int(nSamp * 0.7)

# Create a random number generator
rng = np.random.default_rng()

# Generate a random permutation of the sample indices
# This ensures that the samples are randomly ordered before splitting
perms = rng.permutation(nSamp)

# Select 70% of the samples for the training set
Xtr = X[perms[0:nSampTr], :]
ytr = y[perms[0:nSampTr]]

# Select the remaining 30% of the samples for the test set
Xte = X[perms[nSampTr:], :]
yte = y[perms[nSampTr:]]

• Create a linear classifer based on Fisher Linear Discriminant analysis and on the selected training data.

In [20]:
ixSup = np.where(ytr==1)[0]
ixAvg = np.where(ytr==2)[0]

muSup = np.array([np.mean(Xtr[ixSup,:],axis=0)]).T
muAvg = np.array([np.mean(Xtr[ixAvg,:],axis=0)]).T

S1 = (Xtr[ixSup,:].T-muSup)@(Xtr[ixSup,:].T-muSup).T
S2 = (Xtr[ixAvg,:].T-muAvg)@(Xtr[ixAvg,:].T-muAvg).T

Sw = S1+S2 # within-class scatter matrix, which is the sum of the scatter matrices for the "Super" and "Average" classes, representing the total variability within each class
SwInv = np.linalg.inv(Sw) # inverse of the within-class scatter matrix, which is used to compute the linear discriminant function for LDA by weighting the difference between the class means

w = SwInv@(muSup-muAvg) # weight vector

b = -(w.T@muSup+w.T@muAvg) / 2 # bias 

# Plot the training data and the hyperplane

In [21]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=Xtr[ixSup,0],y=Xtr[ixSup,1],mode='markers',name='Super'))
fig.add_trace(go.Scatter(x=Xtr[ixAvg,0],y=Xtr[ixAvg,1],mode='markers',name='Average'))
fig.update_traces(marker=dict(size=10))


x1=np.arange(20,130,1)

x2=-(w[0,0]/w[1,0])*x1 -b/w[1,0] 
fig.add_trace(go.Scatter(x=x1,y=x2.flatten(),
                         mode='lines', line=dict(dash='dash',color="gray",width=4),name='Hyperplane'))

fig.update_xaxes(title_text='N')
fig.update_yaxes(title_text='PRT/10')
fig.update_layout(
    autosize=False,
    width=900,
    height=800)
fig.show()

-Plot the testing data and the computed hyperplane.

In [22]:
ixSupTe=np.where(yte==1)[0]
ixAvgTe=np.where(yte==2)[0]


fig=go.Figure()
fig.add_trace(go.Scatter(x=Xte[ixSupTe,0],y=Xte[ixSupTe,1],mode='markers',name='Super'))
fig.add_trace(go.Scatter(x=Xte[ixAvgTe,0],y=Xte[ixAvgTe,1],mode='markers',name='Average'))
fig.update_traces(marker=dict(size=10))


x1 = np.arange(20,130,1)

x2=-(w[0,0]/w[1,0])*x1-b/w[1,0]
fig.add_trace(go.Scatter(x=x1,y=x2.flatten(),
                         mode='lines', line=dict(dash='dash',color="gray",width=4),name='Hyperplane'))

fig.update_xaxes(title_text='N')
fig.update_yaxes(title_text='PRT/10')
fig.update_layout(
    autosize=False,
    width=900,
    height=800)
fig.show()

• Compute the performance on the training and testing sets

In [23]:
yptr=np.ones(np.shape(ytr)) # vector of predicted labels for the training set
dx = (w.T@Xtr.T+b).flatten() # compute the decision function values for the training samples
yptr[dx<0]=2


TPTr=np.shape(np.where((ytr[ixSup]==yptr[ixSup]))[0])[0]
TNTr=np.shape(np.where((ytr[ixAvg]==yptr[ixAvg]))[0])[0]
FPTr=np.shape(np.where((ytr[ixAvg]!=yptr[ixAvg]))[0])[0]
FNTr=np.shape(np.where((ytr[ixSup]!=yptr[ixSup]))[0])[0]


SSTr=TPTr/(TPTr+FNTr)
SPTr=TNTr/(TNTr+FPTr)
PRTr=TPTr/(TPTr+FPTr)
F1ScoreTr=2*(PRTr*SSTr)/(PRTr+SSTr)
ACTr=(TNTr+TPTr)/(TPTr+TNTr+FPTr+FNTr)

print("-----------Training----------------")
print("Sensitivity(%)="+str(SSTr*100))
print("Specificity(%)="+str(SPTr*100))
print("Precision(%)="+str(PRTr*100))
print("F1Score(%)="+str(F1ScoreTr*100))
print("Accuracy(%)="+str(ACTr*100))


ypte = np.ones(np.shape(yte)) # vector of predicted labels for the testing set
dx = (w.T@Xte.T+b).flatten() # compute the decision function values for the testing samples
ypte[dx<0] = 2


TPTe = np.shape(np.where((yte[ixSupTe]==ypte[ixSupTe]))[0])[0]
TNTe = np.shape(np.where((yte[ixAvgTe]==ypte[ixAvgTe]))[0])[0]
FPTe = np.shape(np.where((yte[ixAvgTe]!=ypte[ixAvgTe]))[0])[0]
FNTe = np.shape(np.where((yte[ixSupTe]!=ypte[ixSupTe]))[0])[0]


SSTe = TPTe/(TPTe+FNTe)
SPTe = TNTe/(TNTe+FPTe)
PRTe = TPTe/(TPTe+FPTe)
F1ScoreTe = 2*(PRTe*SSTe)/(PRTe+SSTe)
ACTe = (TNTe+TPTe)/(TPTe+TNTe+FPTe+FNTe)

print("-----------Testing----------------")
print("Sensitivity(%)="+str(SSTe*100))
print("Specificity(%)="+str(SPTe*100))
print("Precision(%)="+str(PRTe*100))
print("F1Score(%)="+str(F1ScoreTe*100))
print("Accuracy(%)="+str(ACTe*100))

-----------Training----------------
Sensitivity(%)=100.0
Specificity(%)=79.48717948717949
Precision(%)=79.48717948717949
F1Score(%)=88.57142857142858
Accuracy(%)=88.57142857142857
-----------Testing----------------
Sensitivity(%)=94.73684210526315
Specificity(%)=100.0
Precision(%)=100.0
F1Score(%)=97.2972972972973
Accuracy(%)=96.66666666666667
